# Create grid where each AIS point as centroid of a grid cell. Have it match with GFW grid schema. Convert to UTM Zone 19N. 

## 1. Set Environment

In [2]:
import arcpy
from arcpy import env

env.workspace = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb"
env.overwriteOutput = True

## 2. Input Point Feature Class

In [3]:
gfw_vp_pts = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\gfw_vp_fv_pts"

# confirm spatial ref

desc = arcpy.Describe(gfw_vp_pts)
print(desc.spatialReference.name)

GCS_WGS_1984


## 3. Create a Unique Point Feature Class

In [ ]:
unique_pts = "gfw_vp_unique_pts"

arcpy.management.CopyFeatures(
    in_features=gfw_vp_pts,
    out_feature_class=unique_pts
)

arcpy.management.DeleteIdentical(
    in_dataset=unique_pts,
    fields=["Lon", "Lat"]
)

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\VesselResponses_SNE\\\\VesselResponses_SNE.gdb\\gfw_vp_unique_pts'>

## 4. Create Empty Polygon Feature Class

In [5]:
AOI_sqGrid = "Orsted_sqGridWGS"

arcpy.management.CreateFeatureclass(
    out_path=env.workspace,
    out_name=AOI_sqGrid,
    geometry_type="POLYGON",
    spatial_reference=gfw_vp_pts
)

# carry cell centroids over
arcpy.management.AddField(AOI_sqGrid, "Lon_C", "DOUBLE")
arcpy.management.AddField(AOI_sqGrid, "Lat_C", "DOUBLE")

<Result 'C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\Orsted_sqGridWGS'>

## 5. Construct 0.01 Degree Square Around Each Point

In [6]:
half = 0.005  # half of 0.01 degrees

with arcpy.da.SearchCursor(unique_pts, ["SHAPE@XY", "Lon", "Lat"]) as scur, \
     arcpy.da.InsertCursor(AOI_sqGrid, ["SHAPE@", "Lon_C", "Lat_C"]) as icur:

    for (x, y), lon, lat in scur:

        array = arcpy.Array([
            arcpy.Point(x - half, y - half),  # lower left
            arcpy.Point(x - half, y + half),  # upper left
            arcpy.Point(x + half, y + half),  # upper right
            arcpy.Point(x + half, y - half),  # lower right
            arcpy.Point(x - half, y - half)   # close polygon
        ])

        polygon = arcpy.Polygon(array, arcpy.Describe(unique_pts).spatialReference)

        icur.insertRow([polygon, lon, lat])


## 6. Validate Geometry

In [7]:
# calculate polygon centroids

arcpy.management.FeatureToPoint(
    in_features=AOI_sqGrid,
    out_feature_class="grid_centroids",
    point_location="CENTROID"
)

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\VesselResponses_SNE\\\\VesselResponses_SNE.gdb\\grid_centroids'>

In [8]:
# use near analysis to measure offset

arcpy.analysis.Near(
    in_features="grid_centroids",
    near_features=unique_pts
) # adds a NEAR_DIST field

<Result 'C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\grid_centroids'>

In [9]:
# check for errors

with arcpy.da.SearchCursor("grid_centroids", ["NEAR_DIST"]) as cursor:
    max_dist = max(row[0] for row in cursor)

print(f"Maximum centroid-point distance: {max_dist}") # e-15 value is a safe floating-point artifact

Maximum centroid-point distance: 0.0


Good to move on to UTM projection!

## 7. Define inputs and outputs

In [10]:
# AOI_sqGridWGS already defined
# gfw_vp_pts already defined

AOI_sqGrid_UTM = "Orsted_sqGrid_utm19n" # output polygon
gfw_vp_pts_UTM = "gfw_vp_fv_pts_utm19n" # output point

## 8. Verify Source Spatial Reference

In [11]:
for fc in [AOI_sqGrid, gfw_vp_pts]:
    sr = arcpy.Describe(fc).spatialReference

    print(fc, sr.name, sr.factoryCode)

Orsted_sqGridWGS GCS_WGS_1984 4326
C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\gfw_vp_fv_pts GCS_WGS_1984 4326


## 9. Create Target Spatial Reference (UTM 19N)

In [12]:
utm19n = arcpy.SpatialReference(32619)

## 10. Project the Polygon Grid

In [13]:
arcpy.management.Project(
    in_dataset=AOI_sqGrid,
    out_dataset=AOI_sqGrid_UTM,
    out_coor_system=utm19n,
    transform_method="", # not required for WGS84 to WGS 84
    preserve_shape="PRESERVE_SHAPE"
)

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\VesselResponses_SNE\\\\VesselResponses_SNE.gdb\\Orsted_sqGrid_utm19n'>

## 11. Project the Point Feature Class

In [14]:

arcpy.management.Project(
    in_dataset=gfw_vp_pts,
    out_dataset=gfw_vp_pts_UTM,
    out_coor_system=utm19n,
    transform_method=""
)

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\VesselResponses_SNE\\\\VesselResponses_SNE.gdb\\gfw_vp_fv_pts_utm19n'>

## 12. Validate Geometry

In [15]:
# double check projections

for fc in [AOI_sqGrid_UTM, gfw_vp_pts_UTM]:
    sr = arcpy.Describe(fc).spatialReference

    print(fc, sr.name, sr.linearUnitName)

Orsted_sqGrid_utm19n WGS_1984_UTM_Zone_19N Meter
gfw_vp_fv_pts_utm19n WGS_1984_UTM_Zone_19N Meter


In [16]:
##  confirm centroid-to-centroid alignment

# create polygon centroids

arcpy.management.FeatureToPoint(
    in_features=AOI_sqGrid_UTM,
    out_feature_class="grid_utm_centroids",
    point_location="CENTROID"
)

# measure distance to AIS points

arcpy.analysis.Near(
    in_features="grid_utm_centroids",
    near_features=gfw_vp_pts_UTM
)

# inspect max offset

with arcpy.da.SearchCursor(
    "grid_utm_centroids", ["NEAR_DIST"]
) as cursor:
    max_dist = max(row[0] for row in cursor)

print(f"Maximum centroid‑to‑point offset (meters): {max_dist}")

Maximum centroid‑to‑point offset (meters): 0.0020024987583835536


## 13. Calculate Cell Area of Projected Grid Polygons

In [17]:
# add area field

Area = "Area_m2"

if Area not in [f.name for f in arcpy.ListFields(AOI_sqGrid_UTM)]:
    arcpy.management.AddField(
        AOI_sqGrid_UTM,
        Area,
        "DOUBLE"
    )

In [18]:
# calculate area using shape geometry

arcpy.management.CalculateGeometryAttributes(
    in_features=AOI_sqGrid_UTM,
    geometry_property=[[Area, "AREA"]],
    area_unit="SQUARE_METERS"
)

<Result 'C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\Orsted_sqGrid_utm19n'>

In [19]:
# validate calculations

areas = [row[0] for row in arcpy.da.SearchCursor(AOI_sqGrid_UTM, [Area])]

print(f"Min area (m²): {min(areas)}")
print(f"Max area (m²): {max(areas)}")

# add area in sq km

arcpy.management.AddField(AOI_sqGrid_UTM, "Area_km2", "DOUBLE")

arcpy.management.CalculateField(
    AOI_sqGrid_UTM,
    "Area_km2",
    f"!{Area}! / 1e6",
    "PYTHON3"
)

Min area (m²): 928946.8848023847
Max area (m²): 937379.0488567835


<Result 'C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\Orsted_sqGrid_utm19n'>

In [20]:
# confirm grid cell consistency

import statistics

mean_area = statistics.mean(areas)
stdev_area = statistics.stdev(areas)

print(f"Mean area: {mean_area:.2f} m²")
print(f"Std deviation: {stdev_area:.2f} m²")

Mean area: 933150.57 m²
Std deviation: 2108.00 m²


good to spatial join points and grid cells then move onto STC analyses!

## 14. Spatial Join points and polygons

In [ ]:
# Define inputs and outputs

target = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\Orsted_sqGrid_utm19n"

join = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\gfw_vp_fv_pts_utm19n"

output_fc = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\gfw_vp_fv_joinCellsMany"

In [ ]:
# Execute spatial join

arcpy.analysis.SpatialJoin(
    target_features=target,
    join_features=join,
    out_feature_class=output_fc,
    join_operation="JOIN_ONE_TO_MANY",
    join_type="KEEP_ALL",
    match_option="CONTAINS"
)

In [ ]:
# Verify output

count = int(arcpy.management.GetCount(output_fc)[0])

print(f"Joined records: {count}")